# Collecting Human Preferences for RLHF with the Prolific AI Task Builder

This notebook demonstrates how to use the **Prolific AI Task Builder** to transform LLM-generated responses into structured human preference data — a crucial step in **training and evaluating reward models**.

The **Prolific AI Task Builder** provides a streamlined way to:
- Present multiple model responses to participants
- Collect reliable human preferences at scale
- Export results in a standardized format ready for machine learning workflows

## What This Notebook Does

**Part 1: Generate LLM Responses**
- Load prompts from a file
- Generate multiple responses per prompt using different temperatures
- Create pairwise combinations for human comparison

**Part 2: Collect Human Preferences** 
- Upload response pairs to Prolific
- Create and publish a study for human annotators
- Wait for participants to complete the tasks
- Download and process the results

**Output:** A RLHF dataset in `(prompt, chosen_response, rejected_response)` format ready for reward model training.

---

## Setup

First, we'll import the necessary libraries and our custom modules.

In [41]:
import sys, yaml, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Import our custom modules
from prolific_ai_taskers import (
    ResponseGenerator,          # For generating LLM responses
    ProlificClient,             # For interacting with Prolific API
    load_prompts,               # For loading prompts from JSONL
    create_response_pairs,      # For creating pairwise combinations
    process_preferences         # For processing Prolific responses into RLHF format
)

## Configuration

Load environment variables (API tokens) and configuration settings.

In [2]:
# Load environment variables from .env file
# This file should contain:
#   - HF_TOKEN (Hugging Face token for model access)
#   - PROLIFIC_API_TOKEN (Prolific API token)
#   - PROLIFIC_WORKSPACE_ID (your Prolific workspace)
#   - PROLIFIC_PROJECT_ID (your Prolific project)
load_dotenv()

# Set up file paths
config_path = Path('../examples/config.yaml')
prompts_path = Path('../examples/prompts.jsonl')
output_dir = Path('./outputs')
output_dir.mkdir(exist_ok=True)

print("✅ Paths configured")

✅ Paths configured


In [3]:
# Load configuration from YAML file
# This contains model settings and Prolific study parameters
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

print("Model:", cfg['model'])
print("Temperatures:", cfg['temperatures'])
print("Completions per prompt:", cfg['num_completions_per_prompt'])
print("\n✅ Configuration loaded")

Model: meta-llama/Llama-3.2-3B
Temperatures: [0.7, 1.0]
Completions per prompt: 2

✅ Configuration loaded


---

# Part 1: Generate LLM Responses

In this section, we'll generate multiple responses for each prompt. By using different temperatures, we get responses with varying levels of randomness/creativity.

Note: Typically, this step would be performed after fine-tuning (SFT) the model. However, since the goal of this demo is to focus on the preference stage, we’ll skip the SFT step.

## Step 1.1: Load Prompts

Prompts are stored in a JSONL file (one JSON object per line). Each line has a `prompt` field.

In [4]:
# Load prompts from JSONL file
prompts = load_prompts(prompts_path)

print(f"Loaded {len(prompts)} prompts\n")
print("Example prompts:")
for i, p in enumerate(prompts[:3], 1):
    print(f"{i}. {p['prompt']}")

Loaded 4 prompts

Example prompts:
1. What are the main differences between Python and JavaScript?
2. How can I create a budget and stick to it?
3. What is quantum computing and why is it important?


## Step 1.2: Initialize the Response Generator

This loads the LLM model. It will:
- Auto-detect if you have a GPU (CUDA or Apple Metal) or fall back to CPU
- Load the model weights
- Set up the tokenizer

**Note:** This step may take a minute depending on your model size and internet speed.

In [5]:
# Initialize the generator with the model specified in config
generator = ResponseGenerator(model_name=cfg['model'])

✅ Using Apple Silicon GPU (Metal)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded on mps


## Step 1.3: Generate Completions

Now we'll generate multiple responses for each prompt. The generator will:
- Loop through each prompt
- For each temperature setting (e.g., 0.7 and 1.0)
- Generate N completions (e.g., 2 per temperature)

This creates diverse responses we can compare.

**Example:** With 4 prompts, 2 temperatures, and 2 completions each = 16 total responses

**Note:** This may take several minutes depending on the number of prompts and your hardware.

In [6]:
# Generate all completions
df_completions = generator.generate_completions(
    prompts=prompts,
    temperatures=cfg['temperatures'],
    num_completions_per_prompt=cfg['num_completions_per_prompt'],
    max_new_tokens=cfg['max_new_tokens'],
    top_p=cfg['top_p'],
    verbose=True  # Show progress as it generates
)

print(f"\n✅ Generated {len(df_completions)} total completions")


📝 Prompt 1: What are the main differences between Python and JavaScript?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1 done in 21.75s
    ▶️ Generating completion 2 done in 19.13s
  🌡️ Temperature=1.0
    ▶️ Generating completion 1 done in 19.26s
    ▶️ Generating completion 2 done in 16.58s

📝 Prompt 2: How can I create a budget and stick to it?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1 done in 0.69s
    ▶️ Generating completion 2 done in 19.32s
  🌡️ Temperature=1.0
    ▶️ Generating completion 1 done in 19.25s
    ▶️ Generating completion 2 done in 19.19s

📝 Prompt 3: What is quantum computing and why is it important?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1 done in 17.98s
    ▶️ Generating completion 2 done in 19.18s
  🌡️ Temperature=1.0
    ▶️ Generating completion 1 done in 19.20s
    ▶️ Generating completion 2 done in 17.53s

📝 Prompt 4: How do I troubleshoot a slow computer?
  🌡️ Temperature=0.7
    ▶️ Generating completion 1 done in 19.19s
    ▶️ Gen

In [7]:
# Let's look at the first few completions
df_completions[['prompt', 'temperature', 'response']].head()

,prompt,temperature,response
0,What are the main differences between Python a...,0.7,What are the main differences between Python a...
1,What are the main differences between Python a...,0.7,What are the main differences between Python a...
2,What are the main differences between Python a...,1.0,What are the main differences between Python a...
3,What are the main differences between Python a...,1.0,What are the main differences between Python a...
4,How can I create a budget and stick to it?,0.7,How can I create a budget and stick to it? How...


## Step 1.4: Save Completions

Save the raw completions with metadata for future reference.

In [8]:
# Save completions and metadata
csv_path, metadata_path = generator.save_completions(
    df=df_completions,
    output_dir=output_dir,
    config=cfg,
    num_prompts=len(prompts)
)

Wrote CSV → outputs/completions.csv (16 rows)
Wrote metadata → outputs/run_metadata.json


## Step 1.5: Create Response Pairs

For human annotation, we need pairwise comparisons. This step:
- Groups responses by prompt
- Creates all possible pairs (combinations) of responses for each prompt
- Removes the prompt text from the beginning of responses if it was repeated by the model

**Example:** If we have 4 responses for a prompt, we get $4 \choose 2$ $= 6$ pairs to compare

In [9]:
# Create pairwise combinations
pairs_df = create_response_pairs(
    completions_df=df_completions,
    remove_prompt_prefix=True  # Clean up responses
)

print(f"\nCreated {len(pairs_df)} response pairs")

✅ Created 24 response pairs

Created 24 response pairs


In [10]:
# Preview the pairs - this is what annotators will see
pairs_df.sample(3)

,Prompt,Response A,Response B
6,How can I create a budget and stick to it?,How can I make the most of my money? Where can...,Part 2\nby: Laura Adams\nHow to Create a Budge...
22,How do I troubleshoot a slow computer?,I have a Dell computer with Windows 10 and the...,Try our tips.\nWe have all been there; the com...
3,What are the main differences between Python a...,What are the main differences between Python a...,"They’re both popular programming languages, bu..."


In [11]:
# Save pairs to CSV for Prolific upload
pairs_csv_path = output_dir / 'response_pairs.csv'
pairs_df.to_csv(pairs_csv_path, index=False)
print(f"✅ Saved response pairs to {pairs_csv_path}")

✅ Saved response pairs to outputs/response_pairs.csv


---

# Part 2: Collect Human Preferences via Prolific

Now we'll upload our response pairs to Prolific and collect human preferences.

**Prerequisites:**
- Prolific account with API access
- Valid API token, workspace ID, and project ID in your `.env` file
- Sufficient funds in your Prolific account to pay participants

**What happens in this section:**
1. Create a dataset and upload response pairs
2. Configure the annotation task (how participants will see the data) using Prolific's AI Task Builder
3. Create and publish a study
4. Wait for participants to complete the tasks
5. Download and process the results

## Step 2.1: Initialize Prolific Client

This authenticates with Prolific using your credentials from the `.env` file.

In [12]:
# Initialize Prolific client
# This will read credentials from environment variables and authenticate
client = ProlificClient()

✅ Authenticated as Viviana Marquez


## Step 2.2: Create Dataset and Upload Data

A "dataset" in Prolific is a container for your data. We'll:
1. Create a new dataset
2. Upload the response pairs CSV
3. Wait for Prolific to process it

In [13]:
# Create a dataset
dataset_name = cfg['prolific']['batch_name']
dataset_id = client.create_dataset(name=dataset_name)

✅ Created dataset: 019a1e6f-3aa1-774a-8714-6433fa4e142f


In [14]:
# Upload the response pairs CSV
client.upload_dataset_file(dataset_id, pairs_csv_path)

✅ Uploaded response_pairs.csv


In [15]:
# Wait for Prolific to process the dataset 
if not client.wait_for_dataset_ready(dataset_id):
    raise Exception("Dataset processing failed. Check your CSV format.")

Dataset status: UNINITIALISED
Dataset status: READY
✅ Dataset ready


## Step 2.3: Define Task Schema

The task schema defines what participants will see and how they'll respond.

For pairwise preference collection, we show:
- The prompt
- Response A
- Response B
- A choice: "Which response is better?"

In [16]:
# Define how the task will look to participants
task_details = {
    'task_name': cfg['prolific']['task_schema']['task_name'],
    'task_introduction': cfg['prolific']['task_schema']['task_introduction'].replace('\n', ' '),
    'task_steps': cfg['prolific']['task_schema']['task_steps'].replace('\n', ' '),
    
    # Map CSV columns to display labels
    'inputs': [
        {'key': 'prompt', 'label': 'Prompt'},
        {'key': 'response_a', 'label': 'Response A'},
        {'key': 'response_b', 'label': 'Response B'},
    ],
    
    # Define the question participants will answer
    'judgment': {
        'type': 'single_choice',
        'options': [
            {'value': 'A', 'label': 'Choose Response A'},
            {'value': 'B', 'label': 'Choose Response B'},
        ]
    },
    
    # Randomize which response shows as A or B to avoid position bias
    'randomize_inputs': ['Response A', 'Response B'],
    
    # Require a choice (no skipping)
    'validation': {'require_choice': True}
}

print("✅ Task schema defined")

✅ Task schema defined


## Step 2.4: Create and Configure Batch

A "batch" in Prolific groups your tasks together and defines how they're presented.

In [17]:
# Create batch
batch_id = client.create_batch(
    name=cfg['prolific']['batch_name'],
    dataset_id=dataset_id,
    task_details=task_details
)

✅ Created batch: 019a1e6f-7239-72cf-8422-7025523b6bdd


In [18]:
# Add instructions for participants
instructions = [{
    'type': 'multiple_choice',
    'created_by': client.researcher_name,
    'description': cfg['prolific']['task_schema']['task_question'],
    'options': [
        {'label': 'Response A is better', 'value': 'A'},
        {'label': 'Response B is better', 'value': 'B'},
    ]
}]

client.add_batch_instructions(batch_id, instructions)

✅ Added instructions


In [19]:
# Initialize the batch
# tasks_per_group determines how many comparisons each participant does
client.initialize_batch(
    batch_id=batch_id,
    dataset_id=dataset_id,
    tasks_per_group=cfg['prolific']['task_schema']['tasks_per_group']
)

✅ Initialized batch


In [20]:
# Wait for batch to be ready (Prolific processes it in the background)
if not client.wait_for_batch_ready(batch_id):
    raise Exception("Batch setup failed. Check your task configuration.")

Batch status: READY
✅ Batch ready


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/01_batch.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/02_batch.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
  Preview of Prolific AI Task Builder interface after completing Steps 2.2-2.4. Left: List of created batches. Right: Batch configuration showing the task template with sample data and instructions.
</p>


## Step 2.5: Create and Publish Study

A "study" is what participants see and sign up for. It includes:
- Payment amount
- Estimated time
- Participant eligibility filters
- The link to your batch

In [21]:
# Create study
# The AI Taskers filter ensures participants are qualified for comparative reasoning tasks
filters = [{'filter_id': 'comparative-reasoning', 'selected_values': ['0']}]

study_id = client.create_study(
    batch_id=batch_id,
    task_name=cfg['prolific']['task_schema']['task_name'],
    internal_name=cfg['prolific']['study_setup']['internal_name'],
    description=cfg['prolific']['task_schema']['task_introduction'].replace('\n', ' '),
    estimated_completion_time=cfg['prolific']['study_setup']['estimated_completion_time'],
    max_time=cfg['prolific']['study_setup']['max_time'],
    reward=cfg['prolific']['study_setup']['reward'],
    device_compatibility=cfg['prolific']['study_setup']['device_compatibility'],
    filters=filters
)

print(f"\n📊 Study created: {study_id}")
print(f"View in Prolific dashboard: https://app.prolific.com/researcher/workspaces/studies/{study_id}")

✅ Created study: 68fd8dbf3755881a50f2f5dd

📊 Study created: 68fd8dbf3755881a50f2f5dd
View in Prolific dashboard: https://app.prolific.com/researcher/workspaces/studies/68fd8dbf3755881a50f2f5dd


In [22]:
# Update study with correct participant count
# This ensures each response pair is rated by the specified number of people
client.update_study_participants(
    study_id,
    cfg['prolific']['study_setup']['participants_per_task']
)

✅ Updated study participants


In [23]:
# Publish the study!
# After this, participants can see and sign up for your study
client.publish_study(study_id)

print("\n🎉 Study is now live!")
print("Participants can start working on it.")

✅ Published study

🎉 Study is now live!
Participants can start working on it.


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/03_study.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
 Creator view in Prolific after publishing the study (Step 2.5). Your published study appears in the studies list with its status, participant allocation, and study details.
</p>


<p style="text-align:center; margin-bottom:4px;">
  <img src="img/04_participant.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/05_participant.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; margin-bottom:4px;">
  <img src="img/06_participant.png" alt="Batch 1" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-right:5px;">
  <img src="img/07_participant.png" alt="Batch 2" width="45%" style="border:2px solid #007BFF; border-radius:8px; margin-left:5px;">
</p>

<p style="text-align:center; font-style:italic; color:#555; margin-top:0; font-size:0.9em;">
  What participants see on their end (clockwise from top-left): <br>1) What annotators see when they accept your study, 2) Detailed task instructions and criteria, 3-4) Individual comparison tasks showing a prompt with Response A and Response B, where they select the better response.
</p>

## Step 2.6: Wait for Study Completion

Now we wait for participants to complete the study. 

**You can:**
- Let this cell run (it polls every 60 seconds)
- Stop the notebook and come back later (save the `study_id` and `batch_id`)
- Check progress on the Prolific dashboard

**Note:** The default timeout is 6 hours. You can change it if needed.

In [27]:
# Wait for all participants to complete the study
# This will print status updates every minute
if not client.wait_for_study_completion(study_id, timeout_sec=21600):  # 6 hours
    print("⚠️ Study didn't complete in time. Check Prolific dashboard.")
    print(f"Study ID: {study_id}")
    print(f"Batch ID: {batch_id}")
    print("\nYou can resume later by running the next cells with these IDs.")

Study status: AWAITING REVIEW (15/15 places filled)
Study status: COMPLETED (15/15 places filled)
✅ Study completed!


## Step 2.7: Fetch and Process Results

Once the study is complete, we'll:
1. Download raw responses from all participants
2. Aggregate votes (count how many chose A vs B for each pair)
3. Apply majority voting to determine which response is preferred
4. Create the final RLHF dataset

In [28]:
# Fetch responses from Prolific
df_responses = client.fetch_batch_responses(batch_id)

# Save raw responses
raw_responses_path = output_dir / 'raw_preferences.csv'
df_responses.to_csv(raw_responses_path, index=False)
print(f"Saved raw responses → {raw_responses_path}")

✅ Fetched 24 responses
Saved raw responses → outputs/raw_preferences.csv


In [29]:
# Preview raw responses - each row has responses from all participants
df_responses.head(3)

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Prompt,Response A,Response B,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,...,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp
0,019a1e6f-59e9-709d-aabc-1db86b19f07d,e2fe6a52-bcd1-5bab-8880-01f4dda5c2ce,multiple_choice,"Pick the response that feels overall better, e...",How do I troubleshoot a slow computer?,I have a Dell computer with Windows 10 and the...,It can be due to one of many factors like malw...,66ca1c321a8e2b7f6bf01c79,Response B is better,2025-10-26T03:14:13.264Z,...,2025-10-26T03:18:16.592Z,63bd81465a7246c0da98e4bd,Response B is better,2025-10-26T03:20:04.951Z,6393a3235edc1cd2ebfcb3db,Response B is better,2025-10-26T03:29:49.091Z,65a708e73e47380843931ad8,Response A is better,2025-10-26T04:00:26.537Z
1,019a1e6f-5728-7020-af0b-08068994b9bb,015c8e0f-17c2-5599-bd0b-6d021a2d43b0,multiple_choice,"Pick the response that feels overall better, e...",What are the main differences between Python a...,"Theyâre both popular programming languages, ...",Do they have anything in common?\nJavaScript i...,668daa1bd2c9c15fe7658159,Response B is better,2025-10-26T03:12:24.236Z,...,2025-10-26T03:22:29.508Z,664993e12a5b629050529e17,Response B is better,2025-10-26T03:24:57.182Z,656a3e87ae1fb90ce2539e6a,Response B is better,2025-10-26T03:33:31.102Z,5f5aa7a073b8271a0ab0947c,Response B is better,2025-10-26T04:03:56.178Z
2,019a1e6f-574b-737c-badb-f6035638cf59,015c8e0f-17c2-5599-bd0b-6d021a2d43b0,multiple_choice,"Pick the response that feels overall better, e...",How can I create a budget and stick to it?,How can I make the most of my money? Where can...,Part 2\nby: Laura Adams\nHow to Create a Budge...,668daa1bd2c9c15fe7658159,Response B is better,2025-10-26T03:12:24.236Z,...,2025-10-26T03:22:29.508Z,664993e12a5b629050529e17,Response B is better,2025-10-26T03:24:57.182Z,656a3e87ae1fb90ce2539e6a,Response B is better,2025-10-26T03:33:31.102Z,5f5aa7a073b8271a0ab0947c,Response B is better,2025-10-26T04:03:56.179Z


In [38]:
# Process preferences into RLHF format
# This function:
#   1. Counts votes for each response pair
#   2. Applies majority voting to determine chosen vs rejected
#   3. Saves both the vote counts and final RLHF dataset

df_votes = process_preferences(
    responses_df=df_responses,
    participants_per_task=cfg['prolific']['study_setup']['participants_per_task'],
    output_dir=output_dir
)

Saved votes → outputs/votes_preferences.csv
✅ Created RLHF dataset: 24 pairs (0 ties excluded)
Saved preferences → outputs/preferences.jsonl


In [39]:
# Look at the vote aggregation
df_votes.head(3)

,Prompt,Response A,Response B,A_wins,B_wins
0,How do I troubleshoot a slow computer?,I have a Dell computer with Windows 10 and the...,It can be due to one of many factors like malw...,1,4
1,What are the main differences between Python a...,"Theyâre both popular programming languages, ...",Do they have anything in common?\nJavaScript i...,0,5
2,How can I create a budget and stick to it?,How can I make the most of my money? Where can...,Part 2\nby: Laura Adams\nHow to Create a Budge...,0,5


In [47]:
# Load the RLHF dataset
with open(output_dir / 'preferences.jsonl', 'r') as f:
    rlhf_data = [json.loads(line) for line in f]

# Preview the RLHF dataset
max_chars = 30
truncated_data = []
for item in rlhf_data[:3]:
    truncated_item = item.copy()
    for field in ["chosen_response", "rejected_response"]:
        text = truncated_item[field]
        if len(text) > max_chars:
            truncated_item[field] = text[:max_chars] + "..."
    truncated_data.append(truncated_item)

print(json.dumps(truncated_data, indent=2))

[
  {
    "prompt": "How do I troubleshoot a slow computer?",
    "chosen_response": "It can be due to one of many f...",
    "rejected_response": "I have a Dell computer with Wi..."
  },
  {
    "prompt": "What are the main differences between Python and JavaScript?",
    "chosen_response": "Do they have anything in commo...",
    "rejected_response": "They\u00e2\u0080\u0099re both popular program..."
  },
  {
    "prompt": "How can I create a budget and stick to it?",
    "chosen_response": "Part 2\nby: Laura Adams\nHow to ...",
    "rejected_response": "How can I make the most of my ..."
  }
]


## Step 2.8: Fetch Demographics (Optional)

Prolific provides demographic information about participants. This is useful for:
- Understanding your annotator pool
- Analyzing potential biases
- Reporting in research papers

In [48]:
# Fetch participant demographics
df_demographics = client.fetch_study_demographics(study_id)

# Save demographics
demo_path = output_dir / 'demographic.csv'
df_demographics.to_csv(demo_path, index=False)
print(f"Saved demographics → {demo_path}")

✅ Fetched demographics for 15 participants
Saved demographics → outputs/demographic.csv


In [49]:
# Preview demographics
df_demographics[['Age', 'Sex', 'Country of residence', 'Language']].head()

,Age,Sex,Country of residence,Language
0,33,Female,United States,Russian
2,21,Female,India,Hindi
3,28,Male,United States,English
4,48,Female,United States,English
5,44,Male,United States,English


---

# Final Summary

Let's review what we created and where the files are.

In [51]:
import json

print("="*80)
print("RLHF DATA COLLECTION COMPLETE!")
print("="*80)

print("\n📊 Statistics:")
print(f"  Prompts: {len(prompts)}")
print(f"  Completions generated: {len(df_completions)}")
print(f"  Response pairs: {len(pairs_df)}")
print(f"  Participants: {len(df_demographics)}")

# Count final RLHF pairs
preferences_path = output_dir / 'preferences.jsonl'
with open(preferences_path, 'r') as f:
    rlhf_pairs = sum(1 for _ in f)
print(f"  Final RLHF pairs: {rlhf_pairs}")

print("\n📁 Output files:")
print(f"  {output_dir / 'completions.csv'} - All LLM responses")
print(f"  {output_dir / 'response_pairs.csv'} - Pairwise combinations")
print(f"  {output_dir / 'raw_preferences.csv'} - Raw Prolific responses")
print(f"  {output_dir / 'votes_preferences.csv'} - Vote counts")
print(f"  {output_dir / 'preferences.jsonl'} - 🎯 RLHF dataset (main output!)")
print(f"  {output_dir / 'demographic.csv'} - Participant demographics")
print(f"  {output_dir / 'run_metadata.json'} - Run configuration")

print("\n✅ Next steps:")
print("  1. Use preferences.jsonl to train a reward model")
print("  2. Libraries: Hugging Face TRL, OpenAI PPO, Anthropic Constitutional AI")

print("\n" + "="*80)

RLHF DATA COLLECTION COMPLETE!

📊 Statistics:
  Prompts: 4
  Completions generated: 16
  Response pairs: 24
  Participants: 15
  Final RLHF pairs: 24

📁 Output files:
  outputs/completions.csv - All LLM responses
  outputs/response_pairs.csv - Pairwise combinations
  outputs/raw_preferences.csv - Raw Prolific responses
  outputs/votes_preferences.csv - Vote counts
  outputs/preferences.jsonl - 🎯 RLHF dataset (main output!)
  outputs/demographic.csv - Participant demographics
  outputs/run_metadata.json - Run configuration

✅ Next steps:
  1. Use preferences.jsonl to train a reward model
  2. Libraries: Hugging Face TRL, OpenAI PPO, Anthropic Constitutional AI

